## 第 9 课：tl.dot 与 K 循环累加

题目：[Triton: Block Matrix Multiplication](https://www.deep-ml.com/problems/975?from=Triton%20Essentials)（ID 975）

计算目标：

In [ ]:
c[m, n] = sum_k a[m, k] * b[k, n]

a 形状 `(M, K)`，b 形状 `(K, N)`，c 形状 `(M, N)`。

例如：

In [ ]:
a = [[1, 2],
     [3, 4],
     [5, 6]]
b = [[1, 0],
     [0, 1]]

c = [[1, 2],
     [3, 4],
     [5, 6]]

这是分块矩阵乘法：每个 program 负责 c 的一个 `(BLOCK_M, BLOCK_N)` tile，沿 K 方向迭代、逐步累加。

### 1. 二维 grid：输出 tile 网格

In [ ]:
pid_m = tl.program_id(0)
pid_n = tl.program_id(1)
grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

### 2. fp32 累加器

不管输入是 fp16 还是 fp32，累加器都用 fp32：

In [ ]:
acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

在 K 方向上反复累加，fp16 的舍入误差会越积越多，fp32 精度高得多。

### 3. K 循环 + tl.dot

In [ ]:
for k in range(0, K, BLOCK_K):
    offs_k = k + tl.arange(0, BLOCK_K)

    mask_a = (offs_m[:, None] < M) & (offs_k[None, :] < K)
    mask_b = (offs_k[:, None] < K) & (offs_n[None, :] < N)

    a_tile = tl.load(a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak, mask=mask_a)
    b_tile = tl.load(b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn, mask=mask_b)

    acc += tl.dot(a_tile, b_tile)

- K 不一定是 BLOCK_K 的整数倍，所以循环内要对 k 方向 mask。
- `tl.dot` 是矩阵乘法指令，编译到 GPU 的**张量核心**（Tensor Core），吞吐量远高于逐元素乘加。

### 4. 写回：cast 回输入 dtype

In [ ]:
c_tile = acc.to(a.dtype)  # 输出 dtype 与输入一致
tl.store(c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn, c_tile,
         mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    # TODO 1：取得输出 tile 的行/列 program ID

    # TODO 2：生成输出 tile 的行/列下标 offs_m、offs_n

    # TODO 3：初始化累加器 acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    # 不管输入是 fp16 还是 fp32，累加器都用 fp32

    # TODO 4：K 循环：for k in range(0, K, BLOCK_K):
    #   - offs_k = k + tl.arange(0, BLOCK_K)
    #   - 构造 mask_a、mask_b（注意 k 方向的越界）
    #   - 加载 a_tile (BLOCK_M, BLOCK_K)、b_tile (BLOCK_K, BLOCK_N)
    #   - acc += tl.dot(a_tile, b_tile)

    # TODO 5：把 acc cast 回输入 dtype（acc.to(a.dtype)）

    # TODO 6：带 mask 写入 c（mask 覆盖 M、N 两个方向）
    pass


def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    BLOCK_M = BLOCK_N = BLOCK_K = 32

    # TODO 7：取得 M、K、N（a: (M, K)，b: (K, N)）

    # TODO 8：分配 c（形状 (M, N)，dtype 与 a 相同）

    # TODO 9：创建二维 grid：(cdiv(M, BLOCK_M), cdiv(N, BLOCK_N))

    # TODO 10：启动 kernel
    # stride 使用 a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1)

    # TODO 11：返回 c
    pass

同时回答：

1. `M=64, N=64, K=128`，BLOCK 都是 32 时，grid 是多少？每个 program 的 K 循环执行几次？
2. 为什么累加器固定用 fp32，即使输入是 fp16？
3. `tl.dot` 和手写 `a_tile * b_tile` 再逐元素相加有什么区别？（提示：张量核心）

把代码和三个答案发给我，我继续审查。